This notebook computes the walking times between pairs of grocery stores.

In [64]:
# Initialization
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, LineString, Polygon
from shapely.geometry import MultiPoint
import networkx as nx
import osmnx as ox

In [65]:
# Location of data
datadir = '../Data/'

# On AKB's computer while testing
#datadir = '../../tda-resources-dallas/Dallas/'

In [67]:
#Loading street network

# Our set of grocery stores has points in Collin County 
# (It is City of Dallas, which actually crosses the county line (while also not including non-COD
#     places in Dallas Co)
# This can cause errors when attempting to find path between nodes. 

use_bbox_for_grid = True

if use_bbox_for_grid:
    # To include the southern piece of Collin Co, use bbox
    # bbox = [-97.1,32.5,-96.45,33.1]
    # bbox = [-97.8080, 32.7725, -96.7900, 32.7915]
    coords_list = [-96.808, 32.780, -96.788, 32.807]
    G = ox.graph_from_bbox(coords_list,network_type='walk', simplify=False)
   # G_df = ox.geocode.geocode_to_(bbox,network_type='walk', simplify=False)
else: 
    place = 'University Park, Texas, United States' #!! replace with desired city here
    #G = ox.graph_from_place(place,network_type='walk', buffer_dist = 5000, simplify=False)
    G = ox.graph_from_place(place,network_type='walk', simplify=False)

# coords = [
    
#     (-95.8080, 32.7725),  # Point 1
#       # Point 2
#     (-96.7900, 32.7725),
#       # Point 3
#     (-96.7900, 32.7915),
#     (-96.8080, 32.7915)
#    # Point 4
# ]
four_tuples = [
    (-96.808, 32.780),
    (-96.808, 32.807),
    (-96.788, 32.807),# Top-Left (Northwest)
    (-96.788, 32.780) # Top-Right (Northeast)
     # Bottom-Right (Southeast)
      # Bottom-Left (Southwest)
]

# Create the bounding area polygon
geo_area = Polygon(four_tuples)

geo_area
# print(f"Is this area a valid shape? {geo_area.is_valid}")

# G.draw()

groc_sites = gpd.read_file(datadir+'geo_export_872fcb6c-fbde-4264-ae77-8858a604ed0e.shp') # load grocery store sites

# groc_sites.head()
groc_sites_select = groc_sites[groc_sites['city'] == 'Dallas']
groc_sites_select.head()

groc_sites_select['intersects'] = groc_sites_select.within(geo_area)
groc_sites_select_inner = groc_sites_select[groc_sites_select['intersects'] == True]
groc_sites_select_inner.head()



,address,city,status,store_name,geometry,intersects
2,3524 McKinney Ave,Dallas,Open,Albertsons,POINT (-96.79712 32.80557),True
45,2305 N Central Expy,Dallas,Open,Walmart Market,POINT (-96.79371 32.80061),True
56,2417 N Haskell Rd,Dallas,Open,Target,POINT (-96.79175 32.80481),True
104,4241 Capitol Ave,Dallas,Open,Kroger,POINT (-96.78975 32.80657),True
124,McKinney Ave at Routh St,Dallas,Opening,Whole Foods Market,POINT (-96.80161 32.79554),True


In [68]:
# For each site, find closest "node" in the street graph
nodes=ox.distance.nearest_nodes(G,groc_sites_select_inner['geometry'].x,groc_sites_select_inner['geometry'].y)

In [69]:
nodes

array([ 5882491549,  5879415417,  6677850066, 11764536896,  5851859588])

In [70]:
# Matrix for distance calculations.
walk=np.full((len(groc_sites_select_inner),len(groc_sites_select_inner)),0)

In [71]:
#List pairs of indices for distance computation
#
# Why? If parallel processing, this allows individual computations to be farmed out 
# in arbitrary order
pairs_list = []

# Is the matrix expected to be symmetric? If so, 
# do not calculate d(i,j) and d(j,i) separately.
dist_is_symmetric = True

for i in range(len(nodes)):
    for j in range(i+1,len(nodes)):
        pairs_list.append((i,j))

    if not dist_is_symmetric:
        for j in range(0,i):
            pairs_list.append((i,j))

In [72]:
pairs_list

[(0, 1),
 (0, 2),
 (0, 3),
 (0, 4),
 (1, 2),
 (1, 3),
 (1, 4),
 (2, 3),
 (2, 4),
 (3, 4)]

In [73]:
# compute walk distance (in meters) for each pair of resource sites
mylist = []
for p1,p2 in pairs_list:
    if (p2 == p1 + 1):
        # Just to get a feel how quickly this is going
        print(p1,p2)
    mylist.append(nx.shortest_path_length(G,nodes[p1],nodes[p2],weight='length'))

0 1
1 2
2 3
3 4


In [74]:
#reshape into square array
for i in range(len(pairs_list)):
    walk[pairs_list[i][0],pairs_list[i][1]]=mylist[i]
    if dist_is_symmetric:
        walk[pairs_list[i][1],pairs_list[i][0]]=mylist[i]

In [75]:
dal_walk_distance_meters = walk

In [76]:
walk_speed = 1.42 # meters/sec

In [77]:
dal_walk_distance_seconds = dal_walk_distance_meters/walk_speed #compute matrix of walk TIMES

In [78]:
dal_walk_distance_seconds

array([[   0.        ,  747.18309859,  673.23943662, 1000.        ,
         930.28169014],
       [ 747.18309859,    0.        ,  471.83098592,  791.54929577,
         794.36619718],
       [ 673.23943662,  471.83098592,    0.        ,  326.76056338,
        1176.05633803],
       [1000.        ,  791.54929577,  326.76056338,    0.        ,
        1496.47887324],
       [ 930.28169014,  794.36619718, 1176.05633803, 1496.47887324,
           0.        ]])

In [79]:
np.savez('dal_walk_distance_data', dal_walk_distance_meters = dal_walk_distance_meters, 
         walk_speed = walk_speed, dal_walk_distance_seconds = dal_walk_distance_seconds)